In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('dirty_cafe_sales.csv')

# View the first 5 rows
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# Check data types and count missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 1.1 MB


In [3]:
# Clean column names: lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Print updated column names
print("Updated Column Names:")
print(df.columns.tolist())

Updated Column Names:
['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date']


In [4]:
import numpy as np

# Replace placeholder text strings with true NaN values
placeholders = ['UNKNOWN', 'ERROR', 'unknown', 'error', 'N/A', 'n/a', ' ', '']
df = df.replace(placeholders, np.nan)

# Check exact missing value counts per column now
print("Missing values per column after standardization:")
print(df.isnull().sum())

Missing values per column after standardization:
transaction_id         0
item                 969
quantity             479
price_per_unit       533
total_spent          502
payment_method      3178
location            3961
transaction_date     460
dtype: int64


In [5]:
# Convert numeric columns from text to floating-point numbers
numeric_cols = ['quantity', 'price_per_unit', 'total_spent']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert transaction_date to a proper datetime format
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

# Check updated data types
print("Updated Data Types:")
print(df.dtypes)

Updated Data Types:
transaction_id                 str
item                           str
quantity                   float64
price_per_unit             float64
total_spent                float64
payment_method                 str
location                       str
transaction_date    datetime64[us]
dtype: object


In [6]:
# 1. Fill missing total_spent
df['total_spent'] = df['total_spent'].fillna(df['quantity'] * df['price_per_unit'])

# 2. Fill missing quantity
df['quantity'] = df['quantity'].fillna(df['total_spent'] / df['price_per_unit'])

# 3. Fill missing price_per_unit
df['price_per_unit'] = df['price_per_unit'].fillna(df['total_spent'] / df['quantity'])

# Check missing counts for these 3 columns now
print("Missing numeric values remaining:")
print(df[['quantity', 'price_per_unit', 'total_spent']].isnull().sum())

Missing numeric values remaining:
quantity          38
price_per_unit    38
total_spent       40
dtype: int64


In [7]:
# Strip extra leading/trailing whitespace and capitalize text consistently
text_cols = ['item', 'payment_method', 'location']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    # Convert 'Nan' back to actual NaN after string casting
    df[col] = df[col].replace({'Nan': np.nan, 'None': np.nan})

# Fill missing payment_method and location with 'Unknown'
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['location'] = df['location'].fillna('Unknown')

# Check unique values in these columns to ensure consistency
print("--- Unique Items ---")
print(df['item'].value_counts(dropna=False))

print("\n--- Unique Payment Methods ---")
print(df['payment_method'].value_counts(dropna=False))

print("\n--- Unique Locations ---")
print(df['location'].value_counts(dropna=False))

--- Unique Items ---
item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64

--- Unique Payment Methods ---
payment_method
Unknown           3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

--- Unique Locations ---
location
Unknown     3961
Takeaway    3022
In-Store    3017
Name: count, dtype: int64


In [8]:
# Create a mapping dictionary of price_per_unit -> item from known records
price_item_map = df.dropna(subset=['item', 'price_per_unit']).drop_duplicates(subset=['price_per_unit']).set_index('price_per_unit')['item'].to_dict()

print("Discovered Price-to-Item Map:")
print(price_item_map)

# Fill missing 'item' values where 'price_per_unit' is known
df['item'] = df['item'].fillna(df['price_per_unit'].map(price_item_map))

print("\nRemaining missing items count:")
print(df['item'].isnull().sum())

Discovered Price-to-Item Map:
{2.0: 'Coffee', 3.0: 'Cake', 1.0: 'Cookie', 5.0: 'Salad', 4.0: 'Smoothie', 1.5: 'Tea'}

Remaining missing items count:
6


In [9]:
# 1. Check for duplicate transaction IDs
duplicate_count = df['transaction_id'].duplicated().sum()
print(f"Duplicate Transaction IDs: {duplicate_count}")

# 2. Drop any duplicate rows if they exist
df = df.drop_duplicates(subset=['transaction_id']).reset_index(drop=True)

# 3. Drop rows where key numeric columns or dates are still missing
# (Since these ~40 rows cannot be mathematically recovered)
initial_rows = len(df)
df = df.dropna(subset=['quantity', 'price_per_unit', 'total_spent', 'transaction_date']).reset_index(drop=True)
final_rows = len(df)

print(f"Rows dropped: {initial_rows - final_rows}")
print(f"Final Clean Dataset Shape: {df.shape}")

Duplicate Transaction IDs: 0
Rows dropped: 515
Final Clean Dataset Shape: (9485, 8)


In [10]:
# 1. Verify data types and non-null counts
print("--- FINAL DATA INFO ---")
print(df.info())

# 2. Verify top 5 rows
print("\n--- SAMPLE CLEAN ROWS ---")
print(df.head())

# 3. Export clean dataset to CSV
df.to_csv('cleaned_cafe_sales.csv', index=False)
print("\nSUCCESS: 'cleaned_cafe_sales.csv' saved to your folder!")

--- FINAL DATA INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 9485 entries, 0 to 9484
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    9485 non-null   str           
 1   item              9485 non-null   str           
 2   quantity          9485 non-null   float64       
 3   price_per_unit    9485 non-null   float64       
 4   total_spent       9485 non-null   float64       
 5   payment_method    9485 non-null   str           
 6   location          9485 non-null   str           
 7   transaction_date  9485 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 903.8 KB
None

--- SAMPLE CLEAN ROWS ---
  transaction_id    item  quantity  price_per_unit  total_spent  \
0    TXN_1961373  Coffee       2.0             2.0          4.0   
1    TXN_4977031    Cake       4.0             3.0         12.0   
2    TXN_4271903  Cookie       4.0     